In [1]:
!pip install ultralytics -q
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
from pathlib import Path
import xml.etree.ElementTree as ET

# Paths
# IDD dataset
base_path = Path("/kaggle/input/datasets/uditag21/idddetection/IDD_Detection")

# Uploaded weights dataset
weights_path = "/kaggle/input/datasets/uditag21/idd-yolov8m-best/idd_best.pt"
working_path = Path("/kaggle/working/idd_yolo")

(working_path / "labels/train").mkdir(parents=True, exist_ok=True)
(working_path / "labels/val").mkdir(parents=True, exist_ok=True)

annotations_path = base_path / "Annotations"

classes = [
    "person",
    "rider",
    "car",
    "truck",
    "bus",
    "motorcycle",
    "bicycle",
    "autorickshaw",
    "animal",
    "traffic light",
    "traffic sign"
]

def convert_annotation(xml_file, output_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    size = root.find("size")
    w = int(size.find("width").text)
    h = int(size.find("height").text)

    lines_to_write = []

    for obj in root.iter("object"):
        cls = obj.find("name").text

        if cls not in classes:
            continue

        cls_id = classes.index(cls)

        xmlbox = obj.find("bndbox")
        xmin = float(xmlbox.find("xmin").text)
        xmax = float(xmlbox.find("xmax").text)
        ymin = float(xmlbox.find("ymin").text)
        ymax = float(xmlbox.find("ymax").text)

        # Convert to YOLO format
        x_center = ((xmin + xmax) / 2.0) / w
        y_center = ((ymin + ymax) / 2.0) / h
        width = (xmax - xmin) / w
        height = (ymax - ymin) / h

        lines_to_write.append(
            f"{cls_id} {x_center} {y_center} {width} {height}"
        )

    # Only create label file if at least one valid object exists
    if len(lines_to_write) > 0:
        with open(output_file, "w") as f:
            f.write("\n".join(lines_to_write))


def process_split(split_file, split_name):
    with open(base_path / split_file) as f:
        lines = f.read().strip().split()

    for line in lines:
        xml_file = annotations_path / f"{line}.xml"
        label_dest = working_path / f"labels/{split_name}/{line}.txt"

        label_dest.parent.mkdir(parents=True, exist_ok=True)

        if xml_file.exists():
            convert_annotation(xml_file, label_dest)


process_split("train.txt", "train")
process_split("val.txt", "val")

print("Label conversion complete.")

Label conversion complete.


In [3]:
import os

# Create train/val image folders inside /working
train_dir = working_path / "images/train"
val_dir = working_path / "images/val"
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)

def link_images(split_file, target_dir):
    with open(base_path / split_file) as f:
        lines = f.read().strip().split()
    for line in lines:
        # Each line may include subfolder info like "frontFar/BLR-.../001542_r"
        src = base_path / "JPEGImages" / f"{line}.jpg"
        dst = target_dir / f"{line}.jpg"

        # Ensure subfolders exist in target_dir
        dst.parent.mkdir(parents=True, exist_ok=True)

        if src.exists() and not dst.exists():
            os.symlink(src, dst)

# Link train and val images into /working/images
link_images("train.txt", train_dir)
link_images("val.txt", val_dir)

print("Symlinks created for train and val images in /working/images.")

Symlinks created for train and val images in /working/images.


In [4]:
yaml_content = """
path: /kaggle/working/idd_yolo

train: images/train
val: images/val

names:
"""

for i, cls in enumerate(classes):
    yaml_content += f"  {i}: {cls}\n"

with open("/kaggle/working/idd.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)


path: /kaggle/working/idd_yolo

train: images/train
val: images/val

names:
  0: person
  1: rider
  2: car
  3: truck
  4: bus
  5: motorcycle
  6: bicycle
  7: autorickshaw
  8: animal
  9: traffic light
  10: traffic sign



In [5]:
model = YOLO(weights_path)

metrics = model.val(
    data="/kaggle/working/idd.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    save_json=True
)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,846,129 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.5 ms, read: 71.0±10.2 MB/s, size: 471.2 KB)
val: Scanning /kaggle/working/idd_yolo/labels/val/frontFar/BLR-2018-04-16_16-14-27_frontFar... 10199 images, 26 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 10225/10225 386.5it/s 26.5s<0.0s
val: /kaggle/working/idd_yolo/images/val/highquality_16k/15-07-18-upload/0017291.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.4723      1.3486]
val: /kaggle/working/idd_yolo/images/val/highquality_16k/HYD-2018-08-24_13-02-50/0004247.jpg: 1 duplicate labels removed
val: /kaggle/working/idd_yolo/images/val/highquality_16k/HYD-2018-08-24_13-22-50/0007776.jpg: 1 duplicate labels removed
val: /kaggle/working/idd_yolo/images/val/highquality_16k/HYD-2018-08-24_13-22-50/0008384.jpg: 1 duplicate labels removed
val: New cache cr